In [8]:
import pandas as pd

# 1. Load your shiny new master dataset
trends_df = pd.read_csv("master_global_fitness_trends.csv")

# 2. Extract the unique list of your 40 2-letter codes
target_countries_2letter = trends_df['Country'].unique().tolist()
print(f"Extracted {len(target_countries_2letter)} target countries from Trends data.")

Extracted 80 target countries from Trends data.


In [9]:
# Replace with your actual WHO file name
who_file = "WHO-Obesity_data.csv" 

try:
    who_df = pd.read_csv(who_file)
    print("\nWHO Dataset Columns:")
    print(who_df.columns.tolist())
    
    print("\nFirst 3 rows of WHO data:")
    print(who_df.head(3))
except Exception as e:
    print(f"Could not read WHO file: {e}")


WHO Dataset Columns:
['IndicatorCode', 'Indicator', 'ValueType', 'ParentLocationCode', 'ParentLocation', 'Location type', 'SpatialDimValueCode', 'Location', 'Period type', 'Period', 'IsLatestYear', 'Dim1 type', 'Dim1', 'Dim1ValueCode', 'Dim2 type', 'Dim2', 'Dim2ValueCode', 'Dim3 type', 'Dim3', 'Dim3ValueCode', 'DataSourceDimValueCode', 'DataSource', 'FactValueNumericPrefix', 'FactValueNumeric', 'FactValueUoM', 'FactValueNumericLowPrefix', 'FactValueNumericLow', 'FactValueNumericHighPrefix', 'FactValueNumericHigh', 'Value', 'FactValueTranslationID', 'FactComments', 'Language', 'DateModified']

First 3 rows of WHO data:
  IndicatorCode                                          Indicator ValueType  \
0   NCD_BMI_30C  Prevalence of obesity among adults, BMI &Great...   numeric   
1   NCD_BMI_30C  Prevalence of obesity among adults, BMI &Great...   numeric   
2   NCD_BMI_30C  Prevalence of obesity among adults, BMI &Great...   numeric   

  ParentLocationCode   ParentLocation Location type 

In [10]:
!pip install pycountry

In [11]:
import pandas as pd

print("Step 1: Processing WHO Data Averages...")

# 1. Load the raw WHO Obesity Data
who_file = "WHO-obesity_data.csv" 
who_df = pd.read_csv(who_file)

# 2. Keep only the columns that matter
columns_to_keep = ['SpatialDimValueCode', 'Location', 'Period', 'Dim1', 'FactValueNumeric']
who_clean = who_df[columns_to_keep].copy()

# 3. Filter for the 5-year window (2018 to 2022)
years_to_keep = [2018, 2019, 2020, 2021, 2022]
who_clean = who_clean[who_clean['Period'].isin(years_to_keep)]

# 4. Group by Country and Sex, then calculate the mean
who_avg = who_clean.groupby(
    ['SpatialDimValueCode', 'Location', 'Dim1']
)['FactValueNumeric'].mean().reset_index()

# Round to 2 decimal places
who_avg['FactValueNumeric'] = who_avg['FactValueNumeric'].round(2)

# Rename the columns cleanly
who_avg.rename(columns={
    'SpatialDimValueCode': 'Country_3Letter',
    'Location': 'Country_Name',
    'Dim1': 'Sex',
    'FactValueNumeric': '5yr_Avg_Obesity_%'
}, inplace=True)

# 5. Save the intermediate file
intermediate_file = "intermediate_who_avg.csv"
who_avg.to_csv(intermediate_file, index=False)

print(f"Success! Saved intermediate averages to: {intermediate_file}")

Step 1: Processing WHO Data Averages...
Success! Saved intermediate averages to: intermediate_who_avg.csv


In [12]:
import pandas as pd
import pycountry

print("Step 2: Mapping to Google Trends Countries...")

# 1. Load your master Trends dataset and intermediate WHO data
trends_df = pd.read_csv("master_global_fitness_trends.csv")
who_avg_df = pd.read_csv("intermediate_who_avg.csv")

# Extract the unique 40 2-letter codes from Trends
target_countries_2letter = trends_df['Country'].unique().tolist()

# 2. Create the translation dictionary (2-letter to 3-letter codes)
country_mapping = {}
for code in target_countries_2letter:
    try:
        country_mapping[code] = pycountry.countries.get(alpha_2=code).alpha_3
    except Exception as e:
        pass 

mapping_df = pd.DataFrame(list(country_mapping.items()), columns=['Country_2Letter', 'Country_3Letter'])

# 3. The Magic Merge: Attach 2-letter codes and filter down to the 40 countries
who_final = pd.merge(who_avg_df, mapping_df, on='Country_3Letter', how='inner')

# 4. Save the final, mapped dataset
final_output = "clean_mapped_who_obesity.csv"
who_final.to_csv(final_output, index=False)

print(f"Success! Mapped and filtered down to {len(who_final)} demographic rows.")
print(f"Final dataset saved as: {final_output}")

Step 2: Mapping to Google Trends Countries...
Success! Mapped and filtered down to 240 demographic rows.
Final dataset saved as: clean_mapped_who_obesity.csv


In [13]:
# Replace with your actual WHO file name
who_file = "WHO-activity_data.csv" 

try:
    who_df = pd.read_csv(who_file)
    print("\nWHO Dataset Columns:")
    print(who_df.columns.tolist())
    
    print("\nFirst 3 rows of WHO data:")
    print(who_df.head(3))
except Exception as e:
    print(f"Could not read WHO file: {e}")


WHO Dataset Columns:
['IndicatorCode', 'Indicator', 'ValueType', 'ParentLocationCode', 'ParentLocation', 'Location type', 'SpatialDimValueCode', 'Location', 'Period type', 'Period', 'IsLatestYear', 'Dim1 type', 'Dim1', 'Dim1ValueCode', 'Dim2 type', 'Dim2', 'Dim2ValueCode', 'Dim3 type', 'Dim3', 'Dim3ValueCode', 'DataSourceDimValueCode', 'DataSource', 'FactValueNumericPrefix', 'FactValueNumeric', 'FactValueUoM', 'FactValueNumericLowPrefix', 'FactValueNumericLow', 'FactValueNumericHighPrefix', 'FactValueNumericHigh', 'Value', 'FactValueTranslationID', 'FactComments', 'Language', 'DateModified']

First 3 rows of WHO data:
  IndicatorCode                                          Indicator ValueType  \
0       NCD_PAA  Prevalence of insufficient physical activity a...      text   
1       NCD_PAA  Prevalence of insufficient physical activity a...      text   
2       NCD_PAA  Prevalence of insufficient physical activity a...      text   

  ParentLocationCode   ParentLocation Location type 

In [14]:
import pandas as pd

print("Step 1: Processing WHO Inactivity Data...")

# 1. Load the raw WHO Inactivity Data
who_file = "WHO-activity_data.csv" 
who_df = pd.read_csv(who_file)

# 2. Keep the core columns
columns_to_keep = ['SpatialDimValueCode', 'Location', 'Period', 'Dim1', 'FactValueNumeric']
who_clean = who_df[columns_to_keep].copy()

# 3. Filter for the 5-year window (2018 to 2022)
years_to_keep = [2018, 2019, 2020, 2021, 2022]
who_clean = who_clean[who_clean['Period'].isin(years_to_keep)]

# 4. Group by Country and Sex, then calculate the mean
who_avg = who_clean.groupby(
    ['SpatialDimValueCode', 'Location', 'Dim1']
)['FactValueNumeric'].mean().reset_index()

# Round to 2 decimal places
who_avg['FactValueNumeric'] = who_avg['FactValueNumeric'].round(2)

# Rename the columns cleanly
who_avg.rename(columns={
    'SpatialDimValueCode': 'Country_3Letter',
    'Location': 'Country_Name',
    'Dim1': 'Sex',
    'FactValueNumeric': '5yr_Avg_Inactivity_%'
}, inplace=True)

# 5. Save the intermediate file
intermediate_file = "intermediate_who_inactivity.csv"
who_avg.to_csv(intermediate_file, index=False)

print(f"Success! Saved intermediate inactivity averages to: {intermediate_file}")

Step 1: Processing WHO Inactivity Data...
Success! Saved intermediate inactivity averages to: intermediate_who_inactivity.csv


In [15]:
import pandas as pd

who_df = pd.read_csv("WHO-activity_data.csv")
null_count = who_df['FactValueNumeric'].isna().sum()

print(f"Number of empty rows in FactValueNumeric: {null_count}")

Number of empty rows in FactValueNumeric: 0


In [16]:
import pandas as pd
import pycountry

print("Step 2: Mapping Inactivity Data to Google Trends Countries...")

# 1. Load your master Trends dataset and the new intermediate Inactivity data
trends_df = pd.read_csv("master_global_fitness_trends.csv")
who_avg_df = pd.read_csv("intermediate_who_inactivity.csv")

# Extract the exact 40 target 2-letter codes from your Trends data
target_countries_2letter = trends_df['Country'].unique().tolist()

# 2. Create the translation dictionary (2-letter to 3-letter codes)
country_mapping = {}
for code in target_countries_2letter:
    try:
        country_mapping[code] = pycountry.countries.get(alpha_2=code).alpha_3
    except Exception as e:
        pass # Silently skips any minor mapping errors

# Turn the dictionary into a clean mapping DataFrame
mapping_df = pd.DataFrame(list(country_mapping.items()), columns=['Country_2Letter', 'Country_3Letter'])

# 3. The Magic Merge: Attach 2-letter codes and automatically filter out non-target countries
who_final = pd.merge(who_avg_df, mapping_df, on='Country_3Letter', how='inner')

# 4. Save the final, mapped dataset
final_output = "clean_mapped_who_inactivity.csv"
who_final.to_csv(final_output, index=False)

print(f"Success! Mapped and filtered down to {len(who_final)} demographic rows.")
print(f"Final dataset saved as: {final_output}")

Step 2: Mapping Inactivity Data to Google Trends Countries...
Success! Mapped and filtered down to 240 demographic rows.
Final dataset saved as: clean_mapped_who_inactivity.csv


In [17]:
import pandas as pd

# Replace with your actual World Bank file name
wb_file = "API_SP.URB.TOTL.IN.ZS_DS2_en_csv_v2_1412/API_SP.URB.TOTL.IN.ZS_DS2_en_csv_v2_1412.csv" 

try:
    # World Bank CSVs usually have 4 lines of junk metadata at the top
    wb_df = pd.read_csv(wb_file, skiprows=4)
    
    print("World Bank Dataset Columns (First 10):")
    print(wb_df.columns.tolist()[:10]) 
    
    print("\nWorld Bank Dataset Columns (Last 10):")
    print(wb_df.columns.tolist()[-10:])
    
except Exception as e:
    print(f"Could not read World Bank file: {e}")

World Bank Dataset Columns (First 10):
['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965']

World Bank Dataset Columns (Last 10):
['2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Unnamed: 70']


In [18]:
import pandas as pd
import pycountry

print("Wrangling World Bank Urbanization Data...")

# 1. Load the target Google Trends countries to build our filter
trends_df = pd.read_csv("master_global_fitness_trends.csv")
target_countries_2letter = trends_df['Country'].unique().tolist()

# 2. Build the 2-letter to 3-letter translation dictionary
country_mapping = {}
for code in target_countries_2letter:
    try:
        country_mapping[code] = pycountry.countries.get(alpha_2=code).alpha_3
    except Exception as e:
        pass 

mapping_df = pd.DataFrame(list(country_mapping.items()), columns=['Country_2Letter', 'Country_3Letter'])

# 3. Load the raw World Bank data (skipping the 4 junk metadata rows)
wb_file = "API_SP.URB.TOTL.IN.ZS_DS2_en_csv_v2_1412/API_SP.URB.TOTL.IN.ZS_DS2_en_csv_v2_1412.csv" 
wb_df = pd.read_csv(wb_file, skiprows=4)

# 4. Keep ONLY the Country Code and our 5 target years
# We ignore 'Unnamed: 70' and the years we don't need
columns_to_keep = ['Country Code', '2018', '2019', '2020', '2021', '2022']
wb_clean = wb_df[columns_to_keep].copy()

# 5. THE MELT: Flatten the years into a single column
wb_melted = pd.melt(
    wb_clean, 
    id_vars=['Country Code'], 
    var_name='Year', 
    value_name='Urbanization_%'
)

# 6. Group by Country and calculate the 5-year average
wb_avg = wb_melted.groupby('Country Code')['Urbanization_%'].mean().reset_index()
wb_avg['Urbanization_%'] = wb_avg['Urbanization_%'].round(2)

# 7. The Magic Merge: Attach our 2-letter codes and filter to the 40 countries
wb_final = pd.merge(wb_avg, mapping_df, left_on='Country Code', right_on='Country_3Letter', how='inner')

# Clean up the final columns for export
wb_final = wb_final[['Country_2Letter', 'Country_3Letter', 'Urbanization_%']]

# 8. Save the final mapped dataset
final_output = "clean_mapped_wb_urbanization.csv"
wb_final.to_csv(final_output, index=False)

print(f"Success! Wranged World Bank data for {len(wb_final)} target countries.")
print(f"Final dataset saved as: {final_output}")

Wrangling World Bank Urbanization Data...
Success! Wranged World Bank data for 80 target countries.
Final dataset saved as: clean_mapped_wb_urbanization.csv


In [19]:
import pandas as pd

print("Initiating Final Master Merge...")

# 1. Load all four wrangled datasets
trends_df = pd.read_csv("master_global_fitness_trends.csv")
obesity_df = pd.read_csv("clean_mapped_who_obesity.csv")
inactivity_df = pd.read_csv("clean_mapped_who_inactivity.csv")
urban_df = pd.read_csv("clean_mapped_wb_urbanization.csv")

# 2. THE PIVOT: Fix the WHO granularity (Long to Wide)
obesity_pivot = obesity_df.pivot(index='Country_2Letter', columns='Sex', values='5yr_Avg_Obesity_%').reset_index()
obesity_pivot = obesity_pivot.add_prefix('Obesity_')
obesity_pivot.rename(columns={'Obesity_Country_2Letter': 'Country_2Letter'}, inplace=True)

inactivity_pivot = inactivity_df.pivot(index='Country_2Letter', columns='Sex', values='5yr_Avg_Inactivity_%').reset_index()
inactivity_pivot = inactivity_pivot.add_prefix('Inactivity_')
inactivity_pivot.rename(columns={'Inactivity_Country_2Letter': 'Country_2Letter'}, inplace=True)

# 3. Build the Static "Country Profile" Baseline Table
# This creates one clean row per country with all demographic and health stats
country_profiles = pd.merge(urban_df, obesity_pivot, on='Country_2Letter', how='inner')
country_profiles = pd.merge(country_profiles, inactivity_pivot, on='Country_2Letter', how='inner')

# 4. THE GRAND MERGE: Attach static profiles to the dynamic Time-Series data
final_dataset = pd.merge(trends_df, country_profiles, left_on='Country', right_on='Country_2Letter', how='inner')

# Drop the redundant Country_2Letter column to keep it completely clean
final_dataset.drop(columns=['Country_2Letter'], inplace=True)

# 5. Save the final masterpiece
output_name = "final_global_fitness_data.csv"
final_dataset.to_csv(output_name, index=False)

print(f"\nSuccess! The final dataset has {len(final_dataset)} rows and {len(final_dataset.columns)} columns.")
print(f"Saved as: {output_name}")

Initiating Final Master Merge...

Success! The final dataset has 5040 rows and 15 columns.
Saved as: final_global_fitness_data.csv


In [20]:
import pandas as pd

print("Initiating Advanced Raw Data Audit...\n")

# Load raw datasets
who_ob = pd.read_csv("WHO-obesity_data.csv")
who_in = pd.read_csv("WHO-activity_data.csv")
wb = pd.read_csv("API_SP.URB.TOTL.IN.ZS_DS2_en_csv_v2_1412/API_SP.URB.TOTL.IN.ZS_DS2_en_csv_v2_1412.csv", skiprows=4)

# --- 1. THE MISSING DATA DETECTIVE ---
print("--- 1. WORLD BANK MISSING DATA INVESTIGATION ---")
# Find the exact row where 2022 urbanization data is missing
missing_country = wb[wb['2022'].isna()]['Country Name'].tolist()
print(f"The 1 row missing Urbanization data belongs to: {missing_country}\n")

# --- 2. CATEGORICAL CONSISTENCY CHECK ---
print("--- 2. CATEGORICAL CONSISTENCY (WHO Sex column) ---")
# Ensure the WHO datasets don't have messy typos in their demographic labels 
# We expect ['Both sexes' 'Male' 'Female']
print(f"Obesity categories: {who_ob['Dim1'].unique()}")
print(f"Inactivity categories: {who_in['Dim1'].unique()}\n")

# --- 3. UNIQUENESS (DUPLICATE ROW) CHECK ---
print("--- 3. STRUCTURAL UNIQUENESS ---")
# For WHO, a Country + Year + Sex combination should only appear exactly once. 
ob_dupes = who_ob.duplicated(subset=['SpatialDimValueCode', 'Period', 'Dim1']).sum()
in_dupes = who_in.duplicated(subset=['SpatialDimValueCode', 'Period', 'Dim1']).sum()
print(f"Unexpected duplicate rows in WHO Obesity: {ob_dupes}")
print(f"Unexpected duplicate rows in WHO Inactivity: {in_dupes}\n")

# --- 4. VALIDITY BOUNDS CHECK ---
print("--- 4. STATISTICAL BOUNDS (0% to 100%) ---")
# Percentages cannot be negative, and cannot exceed 100.
print(f"Obesity Range:    {who_ob['FactValueNumeric'].min()}% to {who_ob['FactValueNumeric'].max()}%")
print(f"Inactivity Range: {who_in['FactValueNumeric'].min()}% to {who_in['FactValueNumeric'].max()}%")
print(f"Urbanization Range (2022): {wb['2022'].min()}% to {wb['2022'].max()}%")

Initiating Advanced Raw Data Audit...

--- 1. WORLD BANK MISSING DATA INVESTIGATION ---
The 1 row missing Urbanization data belongs to: ['Not classified']

--- 2. CATEGORICAL CONSISTENCY (WHO Sex column) ---
Obesity categories: ['Male' 'Both sexes' 'Female']
Inactivity categories: ['Female' 'Male' 'Both sexes']

--- 3. STRUCTURAL UNIQUENESS ---
Unexpected duplicate rows in WHO Obesity: 0
Unexpected duplicate rows in WHO Inactivity: 0

--- 4. STATISTICAL BOUNDS (0% to 100%) ---
Obesity Range:    0.072% to 80.61%
Inactivity Range: 2.39% to 73.6%
Urbanization Range (2022): 14.6923070071538% to 100.0%


In [21]:
import pandas as pd

print("Initiating Final Master Dataset Audit...\n")

try:
    df = pd.read_csv("final_global_fitness_data.csv")
    
    # 1. Structural Overview
    print("--- 1. STRUCTURAL OVERVIEW ---")
    print(f"Total Rows: {df.shape[0]}")
    print(f"Total Columns: {df.shape[1]}")
    
    # 2. Uniqueness Check
    print("\n--- 2. UNIQUENESS CHECK ---")
    # Checking if any Country + Date combo appears more than once
    duplicates = df.duplicated(subset=['Date', 'Country']).sum()
    print(f"Duplicate Date/Country rows: {duplicates}")
    
    # 3. Completeness Check
    print("\n--- 3. COMPLETENESS CHECK ---")
    total_nulls = df.isna().sum().sum()
    print(f"Total Missing Values: {total_nulls}")
    if total_nulls > 0:
        print("Columns with missing values:")
        print(df.isna().sum()[df.isna().sum() > 0])
        
    # 4. Validity Bounds
    print("\n--- 4. STATISTICAL BOUNDS (Min/Max) ---")
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
    for col in numeric_cols:
        c_min = df[col].min()
        c_max = df[col].max()
        print(f"  - {col}: Min = {c_min}, Max = {c_max}")

except Exception as e:
    print(f"Error reading the final dataset: {e}")

Initiating Final Master Dataset Audit...

--- 1. STRUCTURAL OVERVIEW ---
Total Rows: 5040
Total Columns: 15

--- 2. UNIQUENESS CHECK ---
Duplicate Date/Country rows: 0

--- 3. COMPLETENESS CHECK ---
Total Missing Values: 0

--- 4. STATISTICAL BOUNDS (Min/Max) ---
  - Gym: Min = 3, Max = 100
  - Cycling: Min = 0, Max = 100
  - Running: Min = 6, Max = 100
  - Yoga: Min = 5, Max = 100
  - Aerobics: Min = 0, Max = 97
  - Urbanization_%: Min = 19.66, Max = 100.0
  - Obesity_Both sexes: Min = 1.76, Max = 44.22
  - Obesity_Female: Min = 1.87, Max = 54.18
  - Obesity_Male: Min = 0.76, Max = 40.88
  - Inactivity_Both sexes: Min = 4.75, Max = 64.66
  - Inactivity_Female: Min = 4.42, Max = 72.19
  - Inactivity_Male: Min = 5.1, Max = 61.75


In [24]:
df['Country'].unique()

array(['NL', 'PL', 'CR', 'GR', 'IE', 'GB', 'SE', 'PA', 'VN', 'MX', 'CO',
       'EG', 'DO', 'AU', 'US', 'KZ', 'LK', 'OM', 'CZ', 'LB', 'IN', 'KR',
       'UZ', 'GT', 'AE', 'AT', 'BE', 'JP', 'SG', 'CH', 'ZM', 'QA', 'AR',
       'NP', 'TZ', 'TR', 'PT', 'ES', 'PK', 'CI', 'EC', 'CA', 'DZ', 'ZW',
       'CL', 'NO', 'UG', 'KE', 'NG', 'CN', 'KW', 'PH', 'MA', 'SN', 'CM',
       'ID', 'TH', 'IQ', 'RO', 'GH', 'ZA', 'JO', 'BR', 'BO', 'ET', 'DE',
       'AZ', 'IR', 'TN', 'UY', 'SA', 'DK', 'NZ', 'FI', 'MY', 'PE', 'HU',
       'BD', 'FR', 'IT'], dtype=object)

In [25]:
country_names = []

# Loop through each code and look up the name using pycountry
for code in df['Country'].unique():
    # Get the country object based on the Alpha-2 code
    country = pycountry.countries.get(alpha_2=code)
    
    if country:
        # Append the common name or official name
        # Some countries like 'GB' might return 'United Kingdom'
        country_names.append(country.name)
    else:
        # Just in case a code is missing/invalid
        country_names.append(f"Unknown ({code})")

# Sort the list alphabetically for a professional presentation
country_names.sort()

# Print the list formatted perfectly for Appendix A
print("Appendix A: Target Nations Included in Analysis\n")
for i, name in enumerate(country_names, 1):
    print(f"{i}. {name}")

Appendix A: Target Nations Included in Analysis

1. Algeria
2. Argentina
3. Australia
4. Austria
5. Azerbaijan
6. Bangladesh
7. Belgium
8. Bolivia, Plurinational State of
9. Brazil
10. Cameroon
11. Canada
12. Chile
13. China
14. Colombia
15. Costa Rica
16. Czechia
17. Côte d'Ivoire
18. Denmark
19. Dominican Republic
20. Ecuador
21. Egypt
22. Ethiopia
23. Finland
24. France
25. Germany
26. Ghana
27. Greece
28. Guatemala
29. Hungary
30. India
31. Indonesia
32. Iran, Islamic Republic of
33. Iraq
34. Ireland
35. Italy
36. Japan
37. Jordan
38. Kazakhstan
39. Kenya
40. Korea, Republic of
41. Kuwait
42. Lebanon
43. Malaysia
44. Mexico
45. Morocco
46. Nepal
47. Netherlands
48. New Zealand
49. Nigeria
50. Norway
51. Oman
52. Pakistan
53. Panama
54. Peru
55. Philippines
56. Poland
57. Portugal
58. Qatar
59. Romania
60. Saudi Arabia
61. Senegal
62. Singapore
63. South Africa
64. Spain
65. Sri Lanka
66. Sweden
67. Switzerland
68. Tanzania, United Republic of
69. Thailand
70. Tunisia
71. Türkiye
72

In [27]:
import pycountry

country_names = []

# A dictionary to intercept and fix overly formal ISO names
name_cleanup = {
    "Bolivia, Plurinational State of": "Bolivia",
    "Tanzania, United Republic of": "Tanzania",
    "Korea, Republic of": "South Korea",
    "Iran, Islamic Republic of": "Iran",
    "Viet Nam": "Vietnam",
    "Russian Federation": "Russia",
    "Syrian Arab Republic": "Syria",
    "Venezuela, Bolivarian Republic of": "Venezuela"
}

# Loop through each code and look up the name using pycountry
for code in df['Country'].unique():
    # Get the country object based on the Alpha-2 code
    country = pycountry.countries.get(alpha_2=code)
    
    if country:
        # Check if the official name is in our cleanup dictionary; if so, replace it. 
        # If not, just use the normal country.name
        clean_name = name_cleanup.get(country.name, country.name)
        
        # Append the common name or official name
        # Some countries like 'GB' might return 'United Kingdom'
        country_names.append(clean_name)
    else:
        # Just in case a code is missing/invalid
        country_names.append(f"Unknown ({code})")

# Sort the list alphabetically for a professional presentation
country_names.sort()

# Print the list formatted perfectly for Appendix A
print("Appendix A: Target Nations Included in Analysis\n")
for i, name in enumerate(country_names, 1):
    print(f"{i}. {name}")

Appendix A: Target Nations Included in Analysis

1. Algeria
2. Argentina
3. Australia
4. Austria
5. Azerbaijan
6. Bangladesh
7. Belgium
8. Bolivia
9. Brazil
10. Cameroon
11. Canada
12. Chile
13. China
14. Colombia
15. Costa Rica
16. Czechia
17. Côte d'Ivoire
18. Denmark
19. Dominican Republic
20. Ecuador
21. Egypt
22. Ethiopia
23. Finland
24. France
25. Germany
26. Ghana
27. Greece
28. Guatemala
29. Hungary
30. India
31. Indonesia
32. Iran
33. Iraq
34. Ireland
35. Italy
36. Japan
37. Jordan
38. Kazakhstan
39. Kenya
40. Kuwait
41. Lebanon
42. Malaysia
43. Mexico
44. Morocco
45. Nepal
46. Netherlands
47. New Zealand
48. Nigeria
49. Norway
50. Oman
51. Pakistan
52. Panama
53. Peru
54. Philippines
55. Poland
56. Portugal
57. Qatar
58. Romania
59. Saudi Arabia
60. Senegal
61. Singapore
62. South Africa
63. South Korea
64. Spain
65. Sri Lanka
66. Sweden
67. Switzerland
68. Tanzania
69. Thailand
70. Tunisia
71. Türkiye
72. Uganda
73. United Arab Emirates
74. United Kingdom
75. United States
7